#### The Logistics Stack

##### 1. Dependencies

In [3]:
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime, timedelta

fake = Faker()
print("Logistics Stack Loaded. Ready for 2025 Simulation.")

Logistics Stack Loaded. Ready for 2025 Simulation.


#### Market Configuration

In [4]:
CARRIERS = ["CMA CGM", "MSC", "Maersk"]
ETHIOPIAN_HUBS = ["Mojo Dry Port", "Modjo Hub", "Addis Ababa Kality"]
DESTINATIONS = ["Hamburg (Germany)", "Antwerp (Belgium)", "Trieste (Italy)", "Jeddah (KSA)"]
COFFEE_GRADES = ["Sidama G1 Washed", "Yirgacheffe G2 Natural", "Guji G1 Specialty", "Limu G2"]

def generate_romina_2025_data(n=180):
    data = []
    # Start of the 2025 Export Peak
    start_date = datetime(2025, 1, 1) 
    
    for i in range(n):
        carrier = random.choice(CARRIERS)
        # Standard export weight (19.2 Metric Tons)
        declared_weight = round(random.uniform(19195.0, 19215.0), 2)
        
        # THE PROBLEM: 17% VGM Error Rate
        # This represents the weight mismatch between Mojo paperwork and Djibouti scales
        if random.random() < 0.17:
            vgm_actual = declared_weight + random.uniform(45.0, 180.0) 
        else:
            vgm_actual = declared_weight
            
        ship_date = start_date + timedelta(days=i // 2)
        
        data.append({
            "Booking_Ref": f"ROM25-{random.randint(10000, 99999)}",
            "Carrier": carrier,
            "Origin_Inland": random.choice(ETHIOPIAN_HUBS),
            "Port_of_Loading": "Djibouti",
            "Destination": random.choice(DESTINATIONS),
            "Grade": random.choice(COFFEE_GRADES),
            "Declared_Weight_KG": declared_weight,
            "Actual_VGM_KG": vgm_actual,
            "Booking_Date": ship_date.strftime('%Y-%m-%d'),
            "SI_Deadline": (ship_date + timedelta(days=5)).strftime('%Y-%m-%d')
        })
    return pd.DataFrame(data)

master_2025_df = generate_romina_2025_data()
print(f"Success: Generated {len(master_2025_df)} records for the 2025 Season.")
master_2025_df.head()

Success: Generated 180 records for the 2025 Season.


,Booking_Ref,Carrier,Origin_Inland,Port_of_Loading,Destination,Grade,Declared_Weight_KG,Actual_VGM_KG,Booking_Date,SI_Deadline
0,ROM25-42852,MSC,Addis Ababa Kality,Djibouti,Hamburg (Germany),Yirgacheffe G2 Natural,19205.52,19205.52,2025-01-01,2025-01-06
1,ROM25-18965,CMA CGM,Modjo Hub,Djibouti,Trieste (Italy),Guji G1 Specialty,19204.96,19204.96,2025-01-01,2025-01-06
2,ROM25-85909,Maersk,Addis Ababa Kality,Djibouti,Hamburg (Germany),Yirgacheffe G2 Natural,19214.55,19214.55,2025-01-02,2025-01-07
3,ROM25-59754,CMA CGM,Mojo Dry Port,Djibouti,Trieste (Italy),Sidama G1 Washed,19210.90,19210.90,2025-01-02,2025-01-07
4,ROM25-16743,CMA CGM,Modjo Hub,Djibouti,Hamburg (Germany),Sidama G1 Washed,19204.84,19204.84,2025-01-03,2025-01-08


#### 3-Portal Manual Entry Mess

In [7]:
# 1. CMA CGM (Excel - uses 'BookingID' and 'Verified_Weight')
cma_df = master_2025_df[master_2025_df['Carrier'] == 'CMA CGM'].copy()
cma_df.rename(columns={'Booking_Ref': 'BookingID', 'Actual_VGM_KG': 'Verified_Weight'}, inplace=True)
cma_df.to_excel("../data/raw/CMA_CGM_Jan2025_Portal.xlsx", index=False)

# 2. MSC (CSV - uses 'Shipment_Ref' and 'Port_Scale_KG')
msc_df = master_2025_df[master_2025_df['Carrier'] == 'MSC'].copy()
msc_df.rename(columns={'Booking_Ref': 'Shipment_Ref', 'Actual_VGM_KG': 'Port_Scale_KG'}, inplace=True)
msc_df.to_csv("../data/raw/MSC_Export_Log_Jan2025.csv", index=False)

# 3. Maersk (JSON - uses standard names but different format)
maersk_df = master_2025_df[master_2025_df['Carrier'] == 'Maersk'].copy()
maersk_df.to_json("../data/raw/Maersk_API_Feed_Jan2025.json", orient='records')

print("Raw files (XLSX, CSV, JSON) exported to /data/raw/")

Raw files (XLSX, CSV, JSON) exported to /data/raw/
